## Data Preprocessing

In [ ]:
import json, os, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, log_loss, roc_curve, auc as sk_auc,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight

from scipy.special import softmax

import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    Trainer, TrainingArguments, logging as hf_logging,
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=ConvergenceWarning)
BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent
os.chdir(BASE_DIR)


In [ ]:
df = pd.read_csv("hf://datasets/MichiganNLP/MAiDE-up/all_data.csv")
df.head()


In [ ]:
df_ru = df[df["Review_Language"] == "Russian"].copy()
display("Размер до фильтрации:", len(df))
display("Размер после фильтрации:", len(df_ru))


In [ ]:
df_ru["text"] = (
    df_ru["Upside_Review"].fillna("") + " " + df_ru["Downside_Review"].fillna("")
).str.strip()

df_ru["label"] = df_ru["source"].astype(int)

display(df_ru["label"].value_counts().rename("count").to_frame())
df_ru[["text", "label"]].head()


In [ ]:
df_final = (
    df_ru[["text", "label"]]
    .copy()
    .loc[lambda d: d["text"].str.len() > 10]
    .reset_index(drop=True)   # важно для Dataset.from_pandas
)
display(f"Итоговый корпус: {len(df_final):,} отзывов")

In [ ]:
lengths = df_final["text"].str.len()

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.5)
plt.xlabel("Длина отзыва (символов)")
plt.ylabel("Количество отзывов")
plt.title("Распределение длин отзывов (MAiDE-up, RU)")
plt.tight_layout()
plt.show()

display(lengths.describe().round(1).to_string())

## train / validation / test split

In [ ]:
VAL_SIZE  = 0.15
TEST_SIZE = 0.15

train_val_df, test_df = train_test_split(
    df_final,
    test_size=TEST_SIZE,
    random_state=seed,
    stratify=df_final["label"]
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=VAL_SIZE / (1 - TEST_SIZE),  # ≈ 0.1765 → ровно 15% от общего
    random_state=seed,
    stratify=train_val_df["label"]
)

display(f"{'Split':12s}  {'N':>5s}  {'real':>5s}  {'fake':>5s}  {'%fake':>6s}")
display("-" * 40)
for name, split in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    n, f = len(split), split["label"].sum()
    display(f"{name:12s}  {n:5d}  {n-f:5d}  {f:5d}  {f/n*100:5.1f}%")

Поскольку исходный корпус содержит сбалансированное распределение классов (50/50),
при разбиении применялась стратификация по целевой переменной.
Пропорция разбиения: 70% / 15% / 15%.

## Baseline TF-IDF + Logistic Regression

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9)
X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_val_tfidf   = tfidf.transform(val_df["text"])
X_test_tfidf  = tfidf.transform(test_df["text"])
display(f"TF-IDF матрица (train): {X_train_tfidf.shape}")


In [ ]:
logreg = LogisticRegression(max_iter=2000, random_state=seed)
logreg.fit(X_train_tfidf, train_df["label"])

val_preds_lr = logreg.predict(X_val_tfidf)
display("Validation — TF-IDF + LogReg")
display(classification_report(val_df["label"], val_preds_lr, digits=4))


## TF-IDF + SVM

In [ ]:
svm = CalibratedClassifierCV(
    LinearSVC(random_state=seed, max_iter=5000),
    method="sigmoid", cv=3
)
svm.fit(X_train_tfidf, train_df["label"])

val_preds_svm = svm.predict(X_val_tfidf)
display("Validation — TF-IDF + SVM")
display(classification_report(val_df["label"], val_preds_svm, digits=4))

## RuBERT (cointegrated/rubert-tiny2)

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_tok_dataset(df_split):
    ds = Dataset.from_pandas(df_split[["text", "label"]].reset_index(drop=True))
    ds = ds.map(
        lambda b: tokenizer(b["text"], truncation=True,
                            padding="max_length", max_length=256),
        batched=True,
    )
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

train_tok = make_tok_dataset(train_df)
val_tok   = make_tok_dataset(val_df)
test_tok  = make_tok_dataset(test_df)

In [ ]:
_metric_f1  = evaluate.load("f1")
_metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": _metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       _metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir                  = "rubert_fake_reviews",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 4,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = seed,
    data_seed                   = seed,
    dataloader_pin_memory       = False,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    compute_metrics = compute_metrics,
)
trainer.train()

## Evaluating models on a test set

In [ ]:
test_preds_lr  = logreg.predict(X_test_tfidf)
test_probs_lr  = logreg.predict_proba(X_test_tfidf)[:, 1]

test_preds_svm = svm.predict(X_test_tfidf)
test_probs_svm = svm.predict_proba(X_test_tfidf)[:, 1]

pred_test       = trainer.predict(test_tok)
test_preds_bert = np.argmax(pred_test.predictions, axis=1)
test_probs_bert = softmax(pred_test.predictions, axis=1)[:, 1]  # softmax из логитов!

for name, preds in [
    ("TF-IDF + LogReg", test_preds_lr),
    ("TF-IDF + SVM",    test_preds_svm),
    ("RuBERT",          test_preds_bert),
]:
    display(f"── {name} ──")
    display(classification_report(test_df["label"], preds, digits=4))

## Ensembling (soft voting)

In [ ]:
val_probs_lr   = logreg.predict_proba(X_val_tfidf)[:, 1]
val_probs_svm  = svm.predict_proba(X_val_tfidf)[:, 1]

pred_val_bert  = trainer.predict(val_tok)
val_probs_bert = softmax(pred_val_bert.predictions, axis=1)[:, 1]
y_val          = val_df["label"].to_numpy()

def tune_ensemble(probs_a, probs_b, y_true, n_w=21, n_t=81):
    best_f1, best_w, best_t = -1, 0.5, 0.5
    for w in np.linspace(0, 1, n_w):
        ens = w * probs_a + (1 - w) * probs_b
        for t in np.linspace(0.1, 0.9, n_t):
            f1 = f1_score(y_true, (ens >= t).astype(int))
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t
    return best_w, best_t, best_f1

In [ ]:
best_w_lr, best_t_lr, val_f1_lr = tune_ensemble(val_probs_lr, val_probs_bert, y_val)
display(f"TF-IDF+RuBERT  val → w={best_w_lr:.2f}, t={best_t_lr:.2f}, F1={val_f1_lr:.4f}")

ens_probs_lr_bert = best_w_lr * test_probs_lr + (1 - best_w_lr) * test_probs_bert
ens_preds_lr_bert = (ens_probs_lr_bert >= best_t_lr).astype(int)

display(classification_report(test_df["label"], ens_preds_lr_bert, digits=4))

In [ ]:
best_w_svm, best_t_svm, val_f1_svm = tune_ensemble(val_probs_svm, val_probs_bert, y_val)
display(f"SVM+RuBERT     val → w={best_w_svm:.2f}, t={best_t_svm:.2f}, F1={val_f1_svm:.4f}")

ens_probs_svm_bert = best_w_svm * test_probs_svm + (1 - best_w_svm) * test_probs_bert
ens_preds_svm_bert = (ens_probs_svm_bert >= best_t_svm).astype(int)

display(classification_report(test_df["label"], ens_preds_svm_bert, digits=4))

## Errors analysis

In [ ]:
test_df = test_df.copy()
test_df["pred_lr"]            = test_preds_lr
test_df["pred_svm"]           = test_preds_svm
test_df["pred_bert"]          = test_preds_bert
test_df["pred_ens_lr_bert"]   = ens_preds_lr_bert
test_df["pred_ens_svm_bert"]  = ens_preds_svm_bert

display(f"{'Модель':38s}  {'Ошибок':>7s}  {'Accuracy':>9s}  {'F1':>7s}")
display("-" * 68)
for col, name in [
    ("pred_lr",           "TF-IDF + LogReg"),
    ("pred_svm",          "TF-IDF + SVM"),
    ("pred_bert",         "RuBERT"),
    ("pred_ens_lr_bert",  f"Ensemble LR+BERT  (w={best_w_lr:.2f})"),
    ("pred_ens_svm_bert", f"Ensemble SVM+BERT (w={best_w_svm:.2f})"),
]:
    n_err = (test_df["label"] != test_df[col]).sum()
    display(f"{name:38s}  {n_err:7d}  "
          f"{accuracy_score(test_df['label'], test_df[col]):9.4f}  "
          f"{f1_score(test_df['label'], test_df[col]):7.4f}")

In [ ]:
errors_bert = test_df[test_df["label"] != test_df["pred_bert"]][["text","label","pred_bert"]]
errors_bert.head(5)

## Final metrics

In [ ]:
results = pd.DataFrame([
    {"Model": name,
     "Test accuracy": accuracy_score(test_df["label"], preds),
     "Test F1":       f1_score(test_df["label"], preds)}
    for name, preds in [
        ("TF-IDF + LogReg",                               test_preds_lr),
        ("TF-IDF + SVM",                                  test_preds_svm),
        ("RuBERT",                                        test_preds_bert),
        (f"Ensemble LR+BERT  (w={best_w_lr:.2f},t={best_t_lr:.2f})",   ens_preds_lr_bert),
        (f"Ensemble SVM+BERT (w={best_w_svm:.2f},t={best_t_svm:.2f})", ens_preds_svm_bert),
    ]
]).sort_values("Test F1", ascending=False).reset_index(drop=True)

results.style.format({"Test accuracy": "{:.4f}", "Test F1": "{:.4f}"}) \
             .background_gradient(subset=["Test F1","Test accuracy"], cmap="Greens")

## Visualisation of results

In [ ]:
labels = results["Model"].str.replace(r"\s+\(.*\)", "", regex=True)
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, results["Test F1"],       w, label="F1",       color="#4C72B0")
ax.bar(x + w/2, results["Test accuracy"], w, label="Accuracy", color="#DD8452")

for i, (f1, acc) in enumerate(zip(results["Test F1"], results["Test accuracy"])):
    ax.text(i - w/2, f1  + 0.002, f"{f1:.4f}",  ha="center", va="bottom", fontsize=8)
    ax.text(i + w/2, acc + 0.002, f"{acc:.4f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0.87, 1.02)
ax.set_ylabel("Score")
ax.set_title("Сравнение качества моделей (тестовая выборка)")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
cm_data = [
    ("TF-IDF + LogReg",                    test_preds_lr),
    ("TF-IDF + SVM",                       test_preds_svm),
    ("RuBERT",                             test_preds_bert),
    (f"Ensemble LR+BERT (w={best_w_lr:.2f})",  ens_preds_lr_bert),
    (f"Ensemble SVM+BERT (w={best_w_svm:.2f})", ens_preds_svm_bert),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, (name, preds) in enumerate(cm_data):
    cm = confusion_matrix(test_df["label"], preds)
    ConfusionMatrixDisplay(cm, display_labels=["Real", "Fake"]).plot(ax=axes[i])
    axes[i].set_title(name, fontsize=10)

axes[-1].set_visible(False)   # последняя ячейка пустая
plt.suptitle("Матрицы ошибок (тестовая выборка)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
roc_models = [
    ("TF-IDF + LogReg",                    test_probs_lr,       "#4C72B0"),
    ("TF-IDF + SVM",                       test_probs_svm,      "#DD8452"),
    ("RuBERT",                             test_probs_bert,     "#55A868"),
    (f"Ensemble LR+BERT  (w={best_w_lr:.2f})",  ens_probs_lr_bert,  "#C44E52"),
    (f"Ensemble SVM+BERT (w={best_w_svm:.2f})", ens_probs_svm_bert, "#8172B2"),
]

plt.figure(figsize=(7, 6))
for name, probs, color in roc_models:
    fpr, tpr, _ = roc_curve(test_df["label"], probs)
    plt.plot(fpr, tpr, label=f"{name}  AUC={sk_auc(fpr,tpr):.4f}", color=color)

plt.plot([0,1],[0,1], "k--", linewidth=0.8, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривые всех моделей")
plt.legend(fontsize=8, loc="lower right")
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
log_hist   = pd.DataFrame(trainer.state.log_history)
train_logs = (log_hist.dropna(subset=["loss"])
              .groupby("epoch")[["loss"]].mean())
eval_logs  = (log_hist.dropna(subset=["eval_loss"])
              .groupby("epoch")[["eval_loss","eval_f1","eval_accuracy"]].mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(train_logs.index, train_logs["loss"], marker="o", label="Train Loss", color="#4C72B0")
ax1.plot(eval_logs.index,  eval_logs["eval_loss"], marker="o", label="Val Loss",   color="#DD8452")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss (train vs val)"); ax1.legend(); ax1.grid(alpha=0.4)

ax2.plot(eval_logs.index, eval_logs["eval_f1"],       marker="o", label="Val F1",       color="#55A868")
ax2.plot(eval_logs.index, eval_logs["eval_accuracy"], marker="s", label="Val Accuracy", color="#8172B2", linestyle="--")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("F1 & Accuracy (val)"); ax2.legend(); ax2.grid(alpha=0.4)

plt.suptitle("Кривые обучения RuBERT", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
iters = [1, 2, 5, 10, 20, 50, 100, 500]
train_losses, val_losses = [], []

sw_tr = compute_sample_weight("balanced", y=train_df["label"].to_numpy())
sw_va = compute_sample_weight("balanced", y=val_df["label"].to_numpy())

for m in iters:
    lr_tmp = LogisticRegression(max_iter=m, random_state=seed)
    lr_tmp.fit(X_train_tfidf, train_df["label"])
    train_losses.append(log_loss(train_df["label"], lr_tmp.predict_proba(X_train_tfidf), sample_weight=sw_tr))
    val_losses.append(  log_loss(val_df["label"],   lr_tmp.predict_proba(X_val_tfidf),   sample_weight=sw_va))

plt.figure(figsize=(7, 4))
plt.plot(iters, train_losses, marker="o", label="Train Loss", color="#4C72B0")
plt.plot(iters, val_losses,   marker="o", label="Val Loss",   color="#DD8452")
plt.xscale("log")
plt.xlabel("max_iter (log scale)")
plt.ylabel("Weighted Log Loss")
plt.title("TF-IDF + LogReg: кривые сходимости")
plt.legend(); plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## Export

In [ ]:
import joblib, os

os.chdir(BASE_DIR)
os.makedirs("models/rubert", exist_ok=True)

joblib.dump(tfidf,  "models/tfidf_vectorizer.pkl")
joblib.dump(logreg, "models/logreg_model.pkl")
joblib.dump(svm,    "models/svm_model.pkl")

ensemble_params = {
    "tfidf_rubert": {"w": float(best_w_lr),  "t": float(best_t_lr)},
    "svm_rubert":   {"w": float(best_w_svm), "t": float(best_t_svm)},
}
with open("models/ensemble_params.json", "w") as f:
    json.dump(ensemble_params, f, indent=2)

model.save_pretrained("models/rubert/")
tokenizer.save_pretrained("models/rubert/")